# SignEase: ASL Citizen → hand landmarks

Turns the ASL Citizen videos into the same MediaPipe Holistic landmarks the
SignEase app uses, and saves **only the landmarks** (small files) to Google
Drive. The 46 GB zip stays on this Colab machine and is never copied to Drive.

Run the cells top to bottom (Runtime → Run all). No restart is needed.

If Colab disconnects, just run everything again: finished chunks in Drive are
skipped, so it continues where it stopped.

In [ ]:
# Step 1: a separate Python 3.11 with exactly the MediaPipe version SignEase
# uses (newer MediaPipe dropped the Holistic tracker). Takes 2-5 minutes.
import os, subprocess, sys
# Colab's inline plotting setting breaks matplotlib in the other Python
# (MediaPipe imports it); this also applies to step 5.
os.environ["MPLBACKEND"] = "Agg"
VENV = "/content/mp311"
def run(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode and check:
        print((r.stdout + r.stderr)[-4000:])
        raise SystemExit("This step failed; send Claude the text above.")
    return r
# System libraries OpenCV and MediaPipe load on Linux.
run("apt-get -qq update && apt-get -qq install -y libgl1 libglib2.0-0 libportaudio2", check=False)
run(f"{sys.executable} -m pip install -q uv")
if not os.path.exists(VENV + "/bin/python"):
    run(f"{sys.executable} -m uv venv --python 3.11 {VENV}")
PY = f"--python {VENV}/bin/python"
run(f"{sys.executable} -m uv pip install {PY} numpy==1.26.4 mediapipe==0.10.14")
# MediaPipe pulls in the desktop OpenCV build; keep only the headless one,
# which needs no display libraries.
run(f"{sys.executable} -m uv pip uninstall {PY} opencv-contrib-python", check=False)
run(f"{sys.executable} -m uv pip install {PY} --reinstall-package opencv-contrib-python-headless numpy==1.26.4 opencv-contrib-python-headless==4.11.0.86")
r = run(f"""{VENV}/bin/python -c "import mediapipe as mp, cv2, numpy; mp.solutions.holistic.Holistic; print('ready: mediapipe', mp.__version__, '| opencv', cv2.__version__, '| numpy', numpy.__version__)" """)
print(r.stdout.strip().splitlines()[-1])

In [ ]:
# Step 2: settings. Paste the ASL Citizen download link you got after
# accepting the license (right-click the download button → Copy link).
ZIP_URL = "PASTE_THE_DOWNLOAD_LINK_HERE"

# "ours": only videos of the 250 signs SignEase already knows (do this first).
# "all":  then every other sign too, to grow the vocabulary (much longer; run
#         it in later sessions, it resumes).
PHASE = "ours"

DRIVE_DIR = "/content/drive/MyDrive/SignEase/asl_citizen"
ZIP_PATH = "/content/ASL_Citizen.zip"   # this machine's disk, not Drive
CHUNK = 400                              # videos per saved file
MAX_WIDTH = 640                          # same as the SignEase backend

from google.colab import drive
drive.mount("/content/drive")
import os, shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
print("free disk on this machine: %.0f GB" % (shutil.disk_usage("/content").free / 1e9))

In [ ]:
# Step 3: download the zip to this machine (about 46 GB; resumes if interrupted).
import os, subprocess, zipfile
assert ZIP_URL.startswith("http"), "Paste the download link into ZIP_URL in step 2."
def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return len(z.namelist()) > 0
    except Exception:
        return False
if not zip_ok(ZIP_PATH):
    subprocess.check_call(["curl", "-L", "--fail", "--retry", "5", "-C", "-", "-o", ZIP_PATH, ZIP_URL])
assert zip_ok(ZIP_PATH), "The download is not a valid zip. Check the link (it may have expired) and run again."
print("zip ready: %.1f GB" % (os.path.getsize(ZIP_PATH) / 1e9))

In [ ]:
# Step 4: read the dataset index and decide the order (SignEase's signs first).
# Send Claude this cell's output.
import io, os, re, zipfile, pandas as pd
SIGNEASE_SIGNS = ["TV", "after", "airplane", "all", "alligator", "animal", "another", "any", "apple", "arm", "aunt", "awake", "backyard", "bad", "balloon", "bath", "because", "bed", "bedroom", "bee", "before", "beside", "better", "bird", "black", "blow", "blue", "boat", "book", "boy", "brother", "brown", "bug", "bye", "callonphone", "can", "car", "carrot", "cat", "cereal", "chair", "cheek", "child", "chin", "chocolate", "clean", "close", "closet", "cloud", "clown", "cow", "cowboy", "cry", "cut", "cute", "dad", "dance", "dirty", "dog", "doll", "donkey", "down", "drawer", "drink", "drop", "dry", "dryer", "duck", "ear", "elephant", "empty", "every", "eye", "face", "fall", "farm", "fast", "feet", "find", "fine", "finger", "finish", "fireman", "first", "fish", "flag", "flower", "food", "for", "frenchfries", "frog", "garbage", "gift", "giraffe", "girl", "give", "glasswindow", "go", "goose", "grandma", "grandpa", "grass", "green", "gum", "hair", "happy", "hat", "hate", "have", "haveto", "head", "hear", "helicopter", "hello", "hen", "hesheit", "hide", "high", "home", "horse", "hot", "hungry", "icecream", "if", "into", "jacket", "jeans", "jump", "kiss", "kitty", "lamp", "later", "like", "lion", "lips", "listen", "look", "loud", "mad", "make", "man", "many", "milk", "minemy", "mitten", "mom", "moon", "morning", "mouse", "mouth", "nap", "napkin", "night", "no", "noisy", "nose", "not", "now", "nuts", "old", "on", "open", "orange", "outside", "owie", "owl", "pajamas", "pen", "pencil", "penny", "person", "pig", "pizza", "please", "police", "pool", "potty", "pretend", "pretty", "puppy", "puzzle", "quiet", "radio", "rain", "read", "red", "refrigerator", "ride", "room", "sad", "same", "say", "scissors", "see", "shhh", "shirt", "shoe", "shower", "sick", "sleep", "sleepy", "smile", "snack", "snow", "stairs", "stay", "sticky", "store", "story", "stuck", "sun", "table", "talk", "taste", "thankyou", "that", "there", "think", "thirsty", "tiger", "time", "tomorrow", "tongue", "tooth", "toothbrush", "touch", "toy", "tree", "uncle", "underwear", "up", "vacuum", "wait", "wake", "water", "wet", "weus", "where", "white", "who", "why", "will", "wolf", "yellow", "yes", "yesterday", "yourself", "yucky", "zebra", "zipper"]
def norm(text):
    return re.sub(r"[^a-z]", "", str(text).lower())
ours = {norm(s): s for s in SIGNEASE_SIGNS}

z = zipfile.ZipFile(ZIP_PATH)
names = z.namelist()
videos = {os.path.basename(n): n for n in names if n.lower().endswith((".mp4", ".mov", ".webm", ".avi"))}
tables = []
for n in names:
    if n.lower().endswith(".csv") and not os.path.basename(n).startswith("."):
        t = pd.read_csv(io.BytesIO(z.read(n)))
        cols = {c.lower(): c for c in t.columns}
        vcol = next((cols[c] for c in cols if "video" in c or "file" in c), None)
        gcol = next((cols[c] for c in cols if "gloss" in c), None)
        pcol = next((cols[c] for c in cols if "participant" in c or "signer" in c or "user" in c), None)
        print("index:", n, len(t), "rows | columns:", list(t.columns))
        if vcol and gcol:
            tables.append(pd.DataFrame({
                "video": t[vcol].astype(str).map(os.path.basename),
                "gloss": t[gcol].astype(str),
                "participant": t[pcol].astype(str) if pcol else "",
                "split": os.path.splitext(os.path.basename(n))[0],
            }))
assert tables, "No index CSV with video and gloss columns found; send this output to Claude."
manifest = pd.concat(tables).drop_duplicates("video")
manifest["member"] = manifest.video.map(videos)
missing = manifest.member.isna().sum()
manifest = manifest.dropna(subset=["member"])
manifest["signease_sign"] = manifest.gloss.map(lambda g: ours.get(norm(g)) or ours.get(norm(re.sub(r"\d+$", "", str(g)))) or "")
manifest["priority"] = (manifest.signease_sign == "").astype(int)
manifest = manifest.sort_values(["priority", "gloss", "video"]).reset_index(drop=True)
manifest["order"] = range(len(manifest))
manifest.to_csv(os.path.join(DRIVE_DIR, "manifest.csv"), index=False)
found = sorted(set(manifest.signease_sign) - {""})
print(f"{len(manifest)} videos, {manifest.gloss.nunique()} signs ({missing} index rows without a video)")
print(f"SignEase signs found: {len(found)} of 250, {(manifest.priority == 0).sum()} videos")
print("not found by name (Claude will check these by hand):", sorted(set(SIGNEASE_SIGNS) - set(found)))

In [ ]:
# Step 5: extract landmarks in the step-1 Python. Keep this tab open; it
# prints progress and an ETA after each saved chunk.
import os, subprocess
RUNNER = r'''
# Written by the SignEase notebook (step 5). Runs in the Python 3.11 set up in
# step 1, with the same MediaPipe version and Holistic settings as SignEase.
import csv, multiprocessing, os, shutil, sys, tempfile, time, zipfile
import numpy as np

ROWS = np.array([61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511])
LAYOUT = (("face_landmarks", 468), ("left_hand_landmarks", 21), ("pose_landmarks", 33), ("right_hand_landmarks", 21))
SETTINGS = dict(static_image_mode=False, model_complexity=1, smooth_landmarks=True,
                refine_face_landmarks=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)
ZIP, MAX_WIDTH = None, 640


def init(zip_path, max_width):
    global ZIP, MAX_WIDTH
    ZIP, MAX_WIDTH = zipfile.ZipFile(zip_path), max_width


def extract(member):
    import cv2
    import mediapipe as mp
    path = os.path.join(tempfile.gettempdir(), "v%d%s" % (os.getpid(), os.path.splitext(member)[1]))
    with open(path, "wb") as f:
        f.write(ZIP.read(member))
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if np.isfinite(fps) and 5 <= fps <= 120 else 30.0
    frames, aspect = [], 0.0
    # A fresh tracker per video, as each recording starts one in the browser.
    with mp.solutions.holistic.Holistic(**SETTINGS) as holistic:
        while True:
            ok, image = cap.read()
            if not ok:
                break
            h, w = image.shape[:2]
            aspect = w / h
            if w > MAX_WIDTH:
                image = cv2.resize(image, (MAX_WIDTH, int(h * MAX_WIDTH / w)), interpolation=cv2.INTER_AREA)
            results = holistic.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            frame = np.full((543, 2), np.nan, np.float32)
            offset = 0
            for attr, count in LAYOUT:
                lm = getattr(results, attr)
                if lm:
                    frame[offset:offset + count] = [(p.x, p.y) for p in lm.landmark]
                offset += count
            frames.append(frame[ROWS])
    cap.release()
    os.remove(path)
    xy = np.stack(frames).astype(np.float16) if frames else np.zeros((0, len(ROWS), 2), np.float16)
    return member, xy, fps, aspect


def main():
    zip_path, drive_dir, phase = sys.argv[1], sys.argv[2], sys.argv[3]
    chunk_size, max_width = int(sys.argv[4]), int(sys.argv[5])
    work_dir = os.path.dirname(os.path.abspath(zip_path))
    with open(os.path.join(drive_dir, "manifest.csv"), newline="") as f:
        rows = list(csv.DictReader(f))
    ours = [r for r in rows if r["priority"] == "0"]
    rest = [r for r in rows if r["priority"] != "0"]
    split = lambda part: [part[i:i + chunk_size] for i in range(0, len(part), chunk_size)]
    chunks = split(ours) + (split(rest) if phase == "all" else [])
    chunk_file = lambda c: os.path.join(drive_dir, "chunk_%06d.npz" % int(c[0]["order"]))
    left = sum(len(c) for c in chunks if not os.path.exists(chunk_file(c)))
    print("%d videos to do (%d already saved)" % (left, sum(map(len, chunks)) - left), flush=True)
    start, done = time.time(), 0
    pool = multiprocessing.get_context("spawn").Pool(os.cpu_count(), initializer=init, initargs=(zip_path, max_width))
    try:
        for c in chunks:
            out = chunk_file(c)
            if os.path.exists(out):
                continue
            results = list(pool.imap(extract, [r["member"] for r in c], chunksize=2))
            local = os.path.join(work_dir, os.path.basename(out))
            np.savez_compressed(
                local, video=np.array([os.path.basename(r[0]) for r in results]),
                order=np.array([int(r["order"]) for r in c]), lengths=np.array([len(r[1]) for r in results]),
                xy=np.concatenate([r[1] for r in results]), fps=np.array([r[2] for r in results], np.float32),
                aspect=np.array([r[3] for r in results], np.float32), rows=ROWS)
            shutil.copy(local, out + ".tmp")
            os.replace(out + ".tmp", out)
            os.remove(local)
            done += len(c)
            left -= len(c)
            rate = done / (time.time() - start)
            print("saved %s | %.0f videos/hour | %d left, about %.1f h" % (os.path.basename(out), rate * 3600, left, left / rate / 3600), flush=True)
    finally:
        pool.terminate()
    print("Phase", phase, "finished.", flush=True)


if __name__ == "__main__":
    main()
'''
script = os.path.join(os.path.dirname(os.path.abspath(ZIP_PATH)), "signease_run.py")
with open(script, "w") as f:
    f.write(RUNNER)
proc = subprocess.Popen([VENV + "/bin/python", script, ZIP_PATH, DRIVE_DIR, PHASE, str(CHUNK), str(MAX_WIDTH)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
noise = ("W0000", "I0000", "XNNPACK", "WARNING: All log messages", "GetPrototype", "warnings.warn", "inference_feedback", "landmark_projection")
for line in proc.stdout:
    if not any(n in line for n in noise):
        print(line, end="", flush=True)
assert proc.wait() == 0, "Extraction stopped with an error (see above); send it to Claude."

In [ ]:
# Step 6: status. When the phase is finished, download the whole
# "SignEase/asl_citizen" folder from drive.google.com (right-click → Download)
# and unzip it into SignEase/Ml/training/asl_citizen/ on your Mac.
import glob, os, numpy as np, pandas as pd
manifest = pd.read_csv(os.path.join(DRIVE_DIR, "manifest.csv"))
files = sorted(glob.glob(os.path.join(DRIVE_DIR, "chunk_*.npz")))
count = sum(len(np.load(f)["video"]) for f in files)
size = sum(os.path.getsize(f) for f in files) / 1e9
print(f"{len(files)} chunks, {count} videos, {size:.2f} GB in Drive")
print(f"SignEase signs: {(manifest.priority == 0).sum()} videos | all: {len(manifest)} videos")

In [ ]:
# Step 7: the 42 signs ASL Citizen names differently. Only names that are the
# SAME hand sign are listed (the first one found is used). Runs after step 5.
import os, re, subprocess
SAME_SIGN = {
    "alligator": ["ALLIGATOR", "CROCODILE"], "another": ["ANOTHER", "OTHER"], "beside": ["BESIDE", "NEXTTO", "NEXT"],
    "callonphone": ["CALLONPHONE", "PHONE", "TELEPHONE", "CALL"], "dad": ["DAD", "FATHER", "DADDY"],
    "every": ["EVERY", "EACH"], "eye": ["EYE", "EYES"], "feet": ["FEET", "FOOT"], "food": ["FOOD", "EAT"],
    "garbage": ["GARBAGE", "TRASH"], "glasswindow": ["GLASS", "WINDOW"], "goose": ["GOOSE"],
    "grandma": ["GRANDMA", "GRANDMOTHER"], "grandpa": ["GRANDPA", "GRANDFATHER"], "haveto": ["HAVETO", "MUST", "NEED"],
    "hesheit": ["HESHEIT", "HE", "SHE", "IT"], "into": ["INTO", "IN"], "jeans": ["JEANS"], "kitty": ["KITTY", "KITTEN"],
    "look": ["LOOK", "LOOKAT"], "minemy": ["MINEMY", "MY", "MINE"], "mitten": ["MITTEN", "MITTENS"],
    "mom": ["MOM", "MOTHER", "MOMMY"], "nap": ["NAP"], "noisy": ["NOISY", "NOISE"], "nuts": ["NUTS", "NUT"],
    "owie": ["OWIE", "HURT", "PAIN"], "pajamas": ["PAJAMAS"], "police": ["POLICE", "COP"], "pool": ["POOL"],
    "potty": ["POTTY", "TOILET"], "puppy": ["PUPPY"], "puzzle": ["PUZZLE"], "say": ["SAY"],
    "shhh": ["SHHH", "SHH", "HUSH"], "shoe": ["SHOE", "SHOES"], "sleepy": ["SLEEPY"], "snack": ["SNACK", "SNACKS"],
    "store": ["STORE", "SHOP"], "tooth": ["TOOTH", "TEETH"], "wake": ["WAKE", "WAKEUP"], "weus": ["WEUS", "WE", "US"],
}
key = lambda g: re.sub(r"[^a-z]", "", re.sub(r"\d+$", "", str(g)).lower())
by_key = {}
for g in manifest.gloss.unique():
    by_key.setdefault(key(g), []).append(g)
chosen = {}
for sign, names in SAME_SIGN.items():
    hit = next((n for n in names if key(n) in by_key), None)
    stems = [key(n)[:4] for n in names if len(key(n)) >= 4]
    near = sorted({g for k, gs in by_key.items() for g in gs if any(s in k for s in stems)})[:8]
    if hit:
        chosen[sign] = by_key[key(hit)]
    print(f"{sign:12s} -> {', '.join(chosen.get(sign, ['(none)'])):24s} | similar names: {near}")
extra = manifest[manifest.gloss.isin([g for gs in chosen.values() for g in gs])].copy()
extra["signease_sign"] = extra.gloss.map({g: s for s, gs in chosen.items() for g in gs})
extra["priority"] = 0
print(f"\n{len(chosen)} of {len(SAME_SIGN)} signs matched, {len(extra)} videos to extract")
EXTRA_DIR = os.path.join(DRIVE_DIR, "extra")
os.makedirs(EXTRA_DIR, exist_ok=True)
extra.to_csv(os.path.join(EXTRA_DIR, "manifest.csv"), index=False)
script = os.path.join(os.path.dirname(os.path.abspath(ZIP_PATH)), "signease_run.py")
proc = subprocess.Popen([VENV + "/bin/python", script, ZIP_PATH, EXTRA_DIR, "ours", str(CHUNK), str(MAX_WIDTH)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
noise = ("W0000", "I0000", "XNNPACK", "WARNING: All log messages", "GetPrototype", "warnings.warn", "inference_feedback", "landmark_projection")
for line in proc.stdout:
    if not any(n in line for n in noise):
        print(line, end="", flush=True)
assert proc.wait() == 0, "Extraction stopped with an error (see above); send it to Claude."

In [ ]:
# Step 5b (extract on the Mac instead): stops the Colab extraction and packs
# only the videos of SignEase's signs into one zip in Google Drive.
import os, re, shutil, subprocess, time, zipfile, pandas as pd
subprocess.call("pkill -f signease_run.py", shell=True)  # stop step 5's workers
SAME_SIGN = {
    "alligator": ["ALLIGATOR", "CROCODILE"], "another": ["ANOTHER", "OTHER"], "beside": ["BESIDE", "NEXTTO", "NEXT"],
    "callonphone": ["CALLONPHONE", "PHONE", "TELEPHONE", "CALL"], "dad": ["DAD", "FATHER", "DADDY"],
    "every": ["EVERY", "EACH"], "eye": ["EYE", "EYES"], "feet": ["FEET", "FOOT"], "food": ["FOOD", "EAT"],
    "garbage": ["GARBAGE", "TRASH"], "glasswindow": ["GLASS", "WINDOW"], "goose": ["GOOSE"],
    "grandma": ["GRANDMA", "GRANDMOTHER"], "grandpa": ["GRANDPA", "GRANDFATHER"], "haveto": ["HAVETO", "MUST", "NEED"],
    "hesheit": ["HESHEIT", "HE", "SHE", "IT"], "into": ["INTO", "IN"], "jeans": ["JEANS"], "kitty": ["KITTY", "KITTEN"],
    "look": ["LOOK", "LOOKAT"], "minemy": ["MINEMY", "MY", "MINE"], "mitten": ["MITTEN", "MITTENS"],
    "mom": ["MOM", "MOTHER", "MOMMY"], "nap": ["NAP"], "noisy": ["NOISY", "NOISE"], "nuts": ["NUTS", "NUT"],
    "owie": ["OWIE", "HURT", "PAIN"], "pajamas": ["PAJAMAS"], "police": ["POLICE", "COP"], "pool": ["POOL"],
    "potty": ["POTTY", "TOILET"], "puppy": ["PUPPY"], "puzzle": ["PUZZLE"], "say": ["SAY"],
    "shhh": ["SHHH", "SHH", "HUSH"], "shoe": ["SHOE", "SHOES"], "sleepy": ["SLEEPY"], "snack": ["SNACK", "SNACKS"],
    "store": ["STORE", "SHOP"], "tooth": ["TOOTH", "TEETH"], "wake": ["WAKE", "WAKEUP"], "weus": ["WEUS", "WE", "US"],
}
key = lambda g: re.sub(r"[^a-z]", "", re.sub(r"\d+$", "", str(g)).lower())
by_key = {}
for g in manifest.gloss.unique():
    by_key.setdefault(key(g), []).append(g)
chosen = {}
for sign, names in SAME_SIGN.items():
    hit = next((n for n in names if key(n) in by_key), None)
    stems = [key(n)[:4] for n in names if len(key(n)) >= 4]
    near = sorted({g for k, gs in by_key.items() for g in gs if any(s in k for s in stems)})[:8]
    if hit:
        chosen[sign] = by_key[key(hit)]
    print(f"{sign:12s} -> {', '.join(chosen.get(sign, ['(none)'])):24s} | similar names: {near}")
gloss_to_sign = {g: s for s, gs in chosen.items() for g in gs}
extra = manifest[(manifest.priority != 0) & manifest.gloss.isin(gloss_to_sign)].copy()
extra["signease_sign"] = extra.gloss.map(gloss_to_sign)
subset = pd.concat([manifest[manifest.priority == 0], extra]).reset_index(drop=True)
subset["priority"] = 0
print(f"\n{len(chosen)} of {len(SAME_SIGN)} other-name signs matched (+{len(extra)} videos)")
print(f"packing {len(subset)} videos of {subset.signease_sign.nunique()} SignEase signs", flush=True)

local = "/content/SignEase_ASL_Citizen_subset.zip"
src = zipfile.ZipFile(ZIP_PATH)
with zipfile.ZipFile(local, "w", zipfile.ZIP_STORED, allowZip64=True) as out:
    out.writestr("manifest_all.csv", manifest.to_csv(index=False))
    out.writestr("manifest.csv", subset.to_csv(index=False))
    for i, member in enumerate(subset.member):
        out.writestr(member, src.read(member))
        if i % 1000 == 0:
            print(f"  {i} / {len(subset)}", flush=True)
size = os.path.getsize(local) / 1e9
print(f"zip: {size:.2f} GB, copying to Drive...", flush=True)
start = time.time()
shutil.copy(local, os.path.join(DRIVE_DIR, os.path.basename(local)))
print(f"Done in {time.time() - start:.0f}s: My Drive/SignEase/asl_citizen/{os.path.basename(local)} ({size:.2f} GB)")